In [ ]:
import pandas as pd
import os
import time
from obspy.clients.fdsn import Client
from obspy import UTCDateTime

# --- 1. KONFIGURASI ---
CATALOG_PATH = '/Volumes/Extreme SSD/benchmark_bmkg_indonesia/output/katalog_2001_keatas.csv'
SAVE_DIR = '/Volumes/Extreme SSD/benchmark_bmkg_indonesia/waveform_data_2001'
LOG_FILE = os.path.join(SAVE_DIR, 'download_progress.txt') # File untuk mencatat progres

client = Client("IRIS")

def get_last_processed_row():
    """Membaca baris terakhir yang berhasil diunduh dari file log."""
    if os.path.exists(LOG_FILE):
        with open(LOG_FILE, 'r') as f:
            lines = f.readlines()
            if lines:
                return int(lines[-1].strip())
    return 0

def download_micro_batch(df, num_batches=10, size_per_batch=10):
    # Tentukan mulai dari baris mana berdasarkan log
    start_row = get_last_processed_row()
    print(f"[INFO] Melanjutkan unduhan dari baris: {start_row}")
    
    current_index = start_row
    
    for i in range(num_batches):
        print(f"\n--- Memproses Micro-Batch {i+1}/{num_batches} (Baris: {current_index} ke {current_index + size_per_batch}) ---")
        
        bulk_list = []
        df_subset = df.iloc[current_index : current_index + size_per_batch].copy()
        
        for idx, row in df_subset.iterrows():
            try:
                # Konversi waktu ke format ISO untuk ObsPy
                ts = pd.to_datetime(f"{row['Date']} {row['Time (UTC)']}", dayfirst=True)
                t = UTCDateTime(ts.strftime('%Y-%m-%dT%H:%M:%S'))
                bulk_list.append(("*", "*", "*", "BHZ", t, t + 10))
            except:
                continue

        if bulk_list:
            try:
                # Eksekusi unduh
                st = client.get_waveforms_bulk(bulk_list)
                file_name = f"micro_batch_{current_index}.mseed"
                output_path = os.path.join(SAVE_DIR, file_name)
                
                st.write(output_path, format="MSEED")
                print(f"[SUKSES] Berhasil mengunduh batch baris {current_index}")
                
                # CATAT PROGRES: Simpan baris terakhir yang sukses
                current_index += size_per_batch
                with open(LOG_FILE, 'a') as f:
                    f.write(f"{current_index}\n")
                    
            except Exception as e:
                print(f"[ERROR] Gagal di baris {current_index}. Pesan: {e}")
                # Jika error 413 tetap muncul, kita hentikan agar tidak looping error
                if "413" in str(e):
                    print("[STOP] Server masih menolak ukuran ini. Coba perkecil size_per_batch menjadi 5.")
                    break
        
        time.sleep(5) # Jeda lebih lama untuk keamanan server

# --- EKSEKUSI ---
if os.path.exists(CATALOG_PATH):
    df_all = pd.read_csv(CATALOG_PATH)
    # Kita coba 10 batch kecil (Total 100 event)
    download_micro_batch(df_all, num_batches=10, size_per_batch=10)

In [ ]:
import pandas as pd
import os
import time
from obspy.clients.fdsn import Client
from obspy import UTCDateTime

# --- 1. KONFIGURASI ---
CATALOG_PATH = '/Volumes/Extreme SSD/benchmark_bmkg_indonesia/output/katalog_2001_keatas.csv'
SAVE_DIR = '/Volumes/Extreme SSD/benchmark_bmkg_indonesia/waveform_data_2001'
LOG_FILE = os.path.join(SAVE_DIR, 'download_progress.txt')

client = Client("IRIS")

# Pastikan folder ada
os.makedirs(SAVE_DIR, exist_ok=True)

def get_last_processed_row():
    if os.path.exists(LOG_FILE):
        with open(LOG_FILE, 'r') as f:
            lines = f.readlines()
            if lines: return int(lines[-1].strip())
    return 0

def download_micro_batch_v5(df, num_batches=10, size_per_batch=10):
    start_row = get_last_processed_row()
    print(f"[INFO] Melanjutkan unduhan dari baris: {start_row}")
    
    current_index = start_row
    
    for i in range(num_batches):
        if current_index >= len(df): break
        
        print(f"\n--- Memproses Batch {i+1}/{num_batches} (Indeks: {current_index}) ---")
        
        bulk_list = []
        df_subset = df.iloc[current_index : current_index + size_per_batch].copy()
        
        for idx, row in df_subset.iterrows():
            try:
                # PENALARAN: Mengonversi format "01-Jan-2001 09:12:26"
                # dayfirst=True membantu pandas mengenali tanggal di awal
                datetime_str = f"{row['Date']} {row['Time (UTC)']}"
                ts = pd.to_datetime(datetime_str)
                
                # Konversi ke format ObsPy UTCDateTime
                t = UTCDateTime(ts.isoformat())
                
                # Tambahkan ke daftar bulk (Network, Station, Loc, Chan, Start, End)
                bulk_list.append(("*", "*", "*", "BHZ", t, t + 10))
            except Exception as e:
                print(f"[SKIP] Gagal parsing di baris {idx}: {e}")
                continue

        if bulk_list:
            try:
                # Eksekusi Bulk Download
                st = client.get_waveforms_bulk(bulk_list)
                
                file_name = f"micro_batch_{current_index}.mseed"
                output_path = os.path.join(SAVE_DIR, file_name)
                st.write(output_path, format="MSEED")
                
                print(f"[SUKSES] Berhasil menyimpan: {file_name}")
                
                # Catat progres
                current_index += size_per_batch
                with open(LOG_FILE, 'a') as f:
                    f.write(f"{current_index}\n")
                    
            except Exception as e:
                print(f"[ERROR] Batch {current_index} gagal: {e}")
                if "413" in str(e): break
        
        time.sleep(3) # Jeda untuk keamanan server

# --- EKSEKUSI ---
if os.path.exists(CATALOG_PATH):
    # Membaca CSV (Pastikan separator sesuai, biasanya ',' atau ';')
    df_all = pd.read_csv(CATALOG_PATH)
    download_micro_batch_v5(df_all, num_batches=10, size_per_batch=10)

In [ ]:
import pandas as pd
import os
import time
import warnings
from obspy.clients.fdsn import Client
from obspy import UTCDateTime, Stream
from tqdm.notebook import tqdm

# --- 1. KONFIGURASI ---
CATALOG_PATH = '/Volumes/Extreme SSD/benchmark_bmkg_indonesia/output/katalog_2001_keatas.csv'
BASE_SAVE_DIR = '/Volumes/Extreme SSD/benchmark_bmkg_indonesia/output/waveform_2021_2024'
FAILED_LOG = os.path.join(BASE_SAVE_DIR, 'failed_downloads.txt')

# Timeout dinaikkan ke 120 detik untuk data 3-komponen yang berat
client = Client("IRIS", timeout=120)
warnings.filterwarnings('ignore', category=UserWarning, module='obspy.io.mseed')

def download_zne_ultimate(df, start_year=2001, end_year=2024, batch_size=5):
    os.makedirs(BASE_SAVE_DIR, exist_ok=True)
    df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)
    
    for year in tqdm(range(start_year, end_year + 1), desc="Total Progres"):
        year_dir = os.path.join(BASE_SAVE_DIR, str(year))
        os.makedirs(year_dir, exist_ok=True)
        
        df_year = df[df['Date'].dt.year == year]
        if len(df_year) == 0: continue
            
        # Looping per batch
        for start_idx in tqdm(range(0, len(df_year), batch_size), desc=f"Tahun {year}", leave=False):
            file_name = f"batch_{year}_{start_idx}_3comp.mseed"
            output_path = os.path.join(year_dir, file_name)
            
            # AUTO-RESUME
            if os.path.exists(output_path): continue
            
            subset = df_year.iloc[start_idx : start_idx + batch_size]
            bulk_list = []
            for _, row in subset.iterrows():
                try:
                    t = UTCDateTime(f"{row['Date'].strftime('%Y-%m-%d')} {row['Time (UTC)']}")
                    bulk_list.append(("*", "*", "*", "BH?", t, t + 10))
                except: continue
            
            if bulk_list:
                try:
                    # STRATEGI A: Coba download sekaligus (Bulk 5 event)
                    st = client.get_waveforms_bulk(bulk_list)
                    st.write(output_path, format="MSEED")
                except Exception as e:
                    err_msg = str(e) # Pastikan error jadi string (Fix splitlines error)
                    
                    # STRATEGI B: Jika A gagal karena data kegedean atau timeout, coba 1 per 1
                    if "too much data" in err_msg.lower() or "timeout" in err_msg.lower() or "500" in err_msg:
                        combined_st = Stream()
                        for single_item in bulk_list:
                            try:
                                # Coba download per 1 event
                                tmp_st = client.get_waveforms_bulk([single_item])
                                combined_st += tmp_st
                                time.sleep(1) # Jeda sopan antar event
                            except:
                                continue # Jika 1 event ini memang rusak di server, skip ke event berikutnya
                        
                        # Simpan hasil gabungan jika ada yang berhasil
                        if len(combined_st) > 0:
                            combined_st.write(output_path, format="MSEED")
                        else:
                            with open(FAILED_LOG, 'a') as f:
                                f.write(f"FAILED | Tahun {year} | Idx {subset.index.tolist()} | Error: {err_msg[:100]}\n")
                    else:
                        # Error lain-lain (misal koneksi internet Bapak mati)
                        with open(FAILED_LOG, 'a') as f:
                            f.write(f"FAILED | Tahun {year} | Idx {subset.index.tolist()} | Error: {err_msg[:100]}\n")
            
            time.sleep(1.5)

# --- 2. EKSEKUSI ---
if os.path.exists(CATALOG_PATH):
    df_full = pd.read_csv(CATALOG_PATH)
    download_zne_ultimate(df_full)

In [ ]:
from obspy.clients.fdsn import Client
from obspy import UTCDateTime
import json
import os

BASE = "/Volumes/Extreme SSD/benchmark_bmkg_indonesia/output/waveform_2001_2005"
os.makedirs(BASE, exist_ok=True)

client = Client("EARTHSCOPE")

inv = client.get_stations(
    starttime=UTCDateTime("2001-01-01"),
    endtime=UTCDateTime("2005-12-31"),
    channel="BH?,HH?"
)

stations = []
for net in inv:
    for sta in net:
        stations.append((net.code, sta.code))

json.dump(stations, open(f"{BASE}/stations_2001_2005.json","w"))

print("Total stasiun ditemukan:", len(stations))
print("File disimpan di:", f"{BASE}/stations_2001_2005.json")


In [ ]:
from obspy import UTCDateTime
from obspy.clients.fdsn import Client
import pandas as pd
import os, json, time, random, threading
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor

client = Client("IRIS")

# ============================================================
# PATH & KONFIG
# ============================================================

BASE = "/Volumes/Extreme SSD/benchmark_bmkg_indonesia/output/waveform_2001_2005"

event_dir = f"{BASE}/event/"
noise_dir = f"{BASE}/noise/"

os.makedirs(event_dir, exist_ok=True)
os.makedirs(noise_dir, exist_ok=True)

event_log = f"{BASE}/event_done.txt"
noise_log = f"{BASE}/noise_done.txt"
error_log = f"{BASE}/error_log.txt"

open(event_log, "a").close()
open(noise_log, "a").close()
open(error_log, "a").close()

catalog = pd.read_csv("/Volumes/Extreme SSD/benchmark_bmkg_indonesia/output/katalog_indonesia_merged_2001_2005.csv")
stations = json.load(open("/Volumes/Extreme SSD/benchmark_bmkg_indonesia/output/waveform_2001_2005/stations_2001_2005.json"))

channels = ["BHN","BHE","BHZ","HHN","HHE","HHZ"]
MAX_WORKERS = 8  # aman untuk IRIS

log_lock = threading.Lock()

# ============================================================
# HELPER
# ============================================================

def done(logfile, key):
    with log_lock:
        return key in open(logfile).read()

def mark(logfile, key):
    with log_lock:
        with open(logfile, "a") as f:
            f.write(key + "\n")

def log_error(msg):
    with log_lock:
        with open(error_log, "a") as f:
            f.write(msg + "\n")

def safe_download_task(task):
    net, sta, chan, t1, t2, fname = task

    if os.path.exists(fname):
        return True

    for attempt in range(5):
        try:
            st = client.get_waveforms(
                network=net, station=sta, location="*", channel=chan,
                starttime=t1, endtime=t2
            )
            st.write(fname, format="MSEED")
            return True

        except Exception as e:
            log_error(f"[{net}.{sta}.{chan}] {t1} - {t2} | Attempt {attempt+1} | {e}")
            time.sleep(5)

    return False

# ============================================================
# 1. DOWNLOAD EVENT (MULTI-THREAD)
# ============================================================

print("=== Mulai unduh EVENT 3-komponen (multi-thread) ===")

events = catalog.sort_values("time")

for _, row in tqdm(events.iterrows(), total=len(events), desc="Event"):
    try:
        t = UTCDateTime(row["time"])
    except:
        continue

    event_key = str(t)
    if done(event_log, event_key):
        continue

    t1, t2 = t - 300, t + 300

    tasks = []
    for net, sta in stations:
        for chan in channels:
            fname = f"{event_dir}/{event_key}_{net}_{sta}_{chan}.mseed".replace(":","-")
            if not os.path.exists(fname):
                tasks.append((net, sta, chan, t1, t2, fname))

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        list(ex.map(safe_download_task, tasks))

    mark(event_log, event_key)

print("=== EVENT selesai ===")

# ============================================================
# 2. DOWNLOAD NOISE (MULTI-THREAD)
# ============================================================

print("\n=== Mulai unduh NOISE 3-komponen (multi-thread) ===")

years = [2001, 2002, 2003, 2004, 2005]
noise_days = [(y, m, d) for y in years for m in range(1, 13) for d in range(1, 29)]

for year, month, day in tqdm(noise_days, desc="Noise"):
    noise_key = f"{year}-{month}-{day}"

    if done(noise_log, noise_key):
        continue

    t = UTCDateTime(f"{year}-{month:02d}-{day:02d}T00:00:00")
    t1, t2 = t, t + 300

    tasks = []
    for net, sta in stations:
        for chan in channels:
            fname = f"{noise_dir}/{noise_key}_{net}_{sta}_{chan}.mseed"
            if not os.path.exists(fname):
                tasks.append((net, sta, chan, t1, t2, fname))

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        list(ex.map(safe_download_task, tasks))

    mark(noise_log, noise_key)

print("=== NOISE selesai ===")
print("=== Semua unduhan multi-thread, resume otomatis, log lengkap ===")


In [ ]:
from obspy.clients.fdsn import Client
from obspy import UTCDateTime
import json, os

# ============================================================
# KONFIGURASI
# ============================================================

BASE = "/Volumes/Extreme SSD/benchmark_bmkg_indonesia/output/waveform_2001_2005"
os.makedirs(BASE, exist_ok=True)

client = Client("EARTHSCOPE")  # pengganti IRIS

# File input stasiun awal (hasil get_stations sebelumnya)
STATION_FILE = f"{BASE}/stations_2001_2005.json"

# File output
VALID_STATION_FILE = f"{BASE}/valid_stations_2001_2005.json"
VALID_CHANNEL_FILE = f"{BASE}/valid_channels_2001_2005.json"

# Periode data
T1 = UTCDateTime("2001-01-01")
T2 = UTCDateTime("2005-12-31")

# Channel yang ingin dicek
CHANNELS = ["BHN","BHE","BHZ","HHN","HHE","HHZ"]

# ============================================================
# 1. FILTER STASIUN AKTIF
# ============================================================

print("\n=== Tahap 1: Filter stasiun aktif 2001–2005 ===")

stations_all = json.load(open(STATION_FILE))
valid_stations = []

for net, sta in stations_all:
    try:
        inv = client.get_stations(
            network=net,
            station=sta,
            starttime=T1,
            endtime=T2,
            channel="BH?,HH?",
            level="channel"
        )
        valid_stations.append((net, sta))
        print(f"VALID: {net}.{sta}")
    except:
        print(f"TIDAK ADA DATA: {net}.{sta}")

json.dump(valid_stations, open(VALID_STATION_FILE, "w"))
print(f"\nTotal stasiun aktif: {len(valid_stations)}")
print(f"Disimpan ke: {VALID_STATION_FILE}")

# ============================================================
# 2. FILTER CHANNEL VALID PER STASIUN
# ============================================================

print("\n=== Tahap 2: Filter channel valid per stasiun ===")

valid_channels = {}

for net, sta in valid_stations:
    key = f"{net}.{sta}"
    valid_channels[key] = []

    for chan in CHANNELS:
        try:
            inv = client.get_stations(
                network=net,
                station=sta,
                channel=chan,
                starttime=T1,
                endtime=T2,
                level="channel"
            )
            valid_channels[key].append(chan)
            print(f"VALID: {key} → {chan}")
        except:
            pass

# Hapus stasiun yang tidak punya channel sama sekali
valid_channels = {k: v for k, v in valid_channels.items() if len(v) > 0}

json.dump(valid_channels, open(VALID_CHANNEL_FILE, "w"))

print(f"\nTotal stasiun dengan channel valid: {len(valid_channels)}")
print(f"Disimpan ke: {VALID_CHANNEL_FILE}")

print("\n=== SELESAI: Stasiun + Channel valid siap dipakai pipeline unduhan ===")
